# Game Evidence Graph: Dataset Quality Checks

Run this notebook cell by cell after generating `data/processed`. It checks the pipeline outputs, validates the generated files, and gives you small inspection helpers for the study ontology, game ontology, mechanic intervention dataset, evidence graph, and review queue.

## 1. Imports and Paths

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src/game_evidence_graph").exists():
            return candidate
    raise RuntimeError("Could not find project root. Launch Jupyter from the repo or notebooks directory.")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "processed"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CLI = PROJECT_ROOT / ".venv" / "bin" / "game-evidence"
CLI = str(CLI) if CLI.exists() else "game-evidence"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CLI:", CLI)

PROJECT_ROOT: /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph
DATA_DIR: /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph/data/processed
CLI: /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph/.venv/bin/game-evidence


## 2. Confirm Required Artifacts Exist

In [2]:
required_files = [
    "study_ontology.json",
    "game_ontology.json",
    "mechanic_intervention_dataset.csv",
    "mechanic_intervention_dataset.jsonl",
    "causal_claim_graph.json",
    "causal_claim_graph.graphml",
    "evidence_edge_table.csv",
    "completeness_report.csv",
    "extraction_audit_report.md",
    "review_queue.jsonl",
    "design_query_examples.json",
]

artifact_rows = []
missing = []
for name in required_files:
    path = DATA_DIR / name
    exists = path.exists()
    if not exists:
        missing.append(name)
    artifact_rows.append({"file": name, "exists": exists, "size_mb": round(path.stat().st_size / 1_000_000, 3) if exists else None})

artifacts = pd.DataFrame(artifact_rows)
display(artifacts)
assert not missing, f"Missing artifacts: {missing}"

,file,exists,size_mb
0,study_ontology.json,True,2.453
1,game_ontology.json,True,0.465
2,mechanic_intervention_dataset.csv,True,7.781
3,mechanic_intervention_dataset.jsonl,True,13.489
4,causal_claim_graph.json,True,6.550
5,causal_claim_graph.graphml,True,6.697
6,evidence_edge_table.csv,True,2.939
7,completeness_report.csv,True,0.000
8,extraction_audit_report.md,True,0.000
9,review_queue.jsonl,True,2.646


## 3. Run Project Validation

In [3]:
result = subprocess.run(
    [CLI, "validate"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "game-evidence validate failed"

Validation passed.



## 4. Load Outputs

In [4]:
study_ontology = json.loads((DATA_DIR / "study_ontology.json").read_text())
game_ontology = json.loads((DATA_DIR / "game_ontology.json").read_text())
graph = json.loads((DATA_DIR / "causal_claim_graph.json").read_text())

df = pd.read_csv(DATA_DIR / "mechanic_intervention_dataset.csv")
edges = pd.read_csv(DATA_DIR / "evidence_edge_table.csv")
completeness = pd.read_csv(DATA_DIR / "completeness_report.csv")

review_items = [json.loads(line) for line in (DATA_DIR / "review_queue.jsonl").read_text().splitlines() if line.strip()]

print("Loaded dataset rows:", len(df))
print("Loaded graph edges:", len(edges))
print("Loaded review items:", len(review_items))

Loaded dataset rows: 5159
Loaded graph edges: 3964
Loaded review items: 5159


## 5. High-Level Counts

In [5]:
summary = {
    "papers": len(study_ontology.get("papers", [])),
    "studies": len(study_ontology.get("studies", [])),
    "games": len(game_ontology.get("games", [])),
    "dataset_rows": len(df),
    "graph_nodes": len(graph.get("nodes", [])),
    "graph_edges": len(graph.get("edges", [])),
    "edge_table_rows": len(edges),
    "review_items": len(review_items),
    "supported_rows": int(df["is_supported_evidence"].astype(bool).sum()),
    "null_game_rows": int(df["game_id"].isna().sum()),
    "named_game_rows": int(df["game_id"].notna().sum()),
}

pd.Series(summary).to_frame("count")

,count
papers,308
studies,369
games,381
dataset_rows,5159
graph_nodes,3006
graph_edges,3964
edge_table_rows,3964
review_items,5159
supported_rows,3964
null_game_rows,927


In [6]:
display(completeness)

,metric,value
0,dataset_c_rows,5159.00
1,unique_papers,299.00
2,unique_studies,161.00
3,unique_interventions,179.00
4,unique_conditions,262.00
5,unique_games,377.00
6,unique_mechanics,16.00
7,unique_mechanic_sets,716.00
8,unique_outcomes,498.00
9,unique_measurements,498.00


## 6. Sanity Checks for Placeholder Games

In [7]:
game_text = (DATA_DIR / "game_ontology.json").read_text()
placeholder_checks = {
    "has_unspecified_video_game": "Unspecified video game" in game_text,
    "has_unspecified_game_interaction": "Unspecified Game Interaction" in game_text,
}
placeholder_checks

{'has_unspecified_video_game': False,
 'has_unspecified_game_interaction': False}

## 7. Dataset Distributions

In [8]:
for col in ["effect_direction", "attribution_level", "claim_type", "review_status", "evidence_strength"]:
    print(f"\n== {col} ==")
    display(df[col].value_counts(dropna=False).to_frame("rows"))


== effect_direction ==


,rows
effect_direction,
positive,2703
not_reported,1168
NaN,679
negative,357
mixed,225
unclear,27



== attribution_level ==


,rows
attribution_level,
mechanic_set_inferred,3964
unsupported_or_overgenerated,1195



== claim_type ==


,rows
claim_type,
reported_intervention_effect,3964
unsupported_or_overgenerated,1195



== review_status ==


,rows
review_status,
needs_review,5159



== evidence_strength ==


,rows
evidence_strength,
randomized_controlled_trial,895
within_subject_experimental,875
quasi_experimental,858
between_subject_observational,675
correlational,626
not_reported,492
qualitative,241
within_subject_observational,197
unclear,166


## 8. Spot-Check Known Papers

In [9]:
spot_cols = [
    "paper_id",
    "paper_title",
    "study_id",
    "game_name",
    "mechanic_name",
    "outcome_canonical",
    "effect_direction",
    "source_quote",
    "is_supported_evidence",
    "review_status",
]

def show_paper(paper_id: str, n: int = 20) -> pd.DataFrame:
    rows = df[df["paper_id"] == paper_id][spot_cols].head(n)
    if rows.empty:
        print(f"No rows found for {paper_id}")
    return rows

for paper_id in ["paper_002", "paper_017", "paper_196"]:
    print(f"\n== {paper_id} ==")
    display(show_paper(paper_id))


== paper_002 ==


,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
244,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,ERP amplitude (right frontal and posterior),positive,Playing an action game resulted in two effects: one that reflected an increase in the amplitude of the ERPs following training over the right frontal and posterior regions that...,True,needs_review
245,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,allocation of attention to happy faces,negative,Playing an action game resulted in two effects: ... and one that reflected a reduction in the allocation of attention to happy faces.,True,needs_review
246,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,slow wave ERP activity (central–parietal and frontal) greater for targets than nontargets,positive,"In contrast, playing a non-action game resulted in changes in slow wave activity over the central–parietal and frontal regions that were greater for targets (i.e., angry and ha...",True,needs_review
247,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,ERP amplitude (right frontal and posterior),positive,Playing an action game resulted in two effects: one that reflected an increase in the amplitude of the ERPs following training over the right frontal and posterior regions that...,True,needs_review
248,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,allocation of attention to happy faces,negative,Playing an action game resulted in two effects: ... and one that reflected a reduction in the allocation of attention to happy faces.,True,needs_review
249,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,slow wave ERP activity (central–parietal and frontal) greater for targets than nontargets,positive,"In contrast, playing a non-action game resulted in changes in slow wave activity over the central–parietal and frontal regions that were greater for targets (i.e., angry and ha...",True,needs_review
250,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,ERP amplitude (right frontal and posterior),positive,Playing an action game resulted in two effects: one that reflected an increase in the amplitude of the ERPs following training over the right frontal and posterior regions that...,True,needs_review
251,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,allocation of attention to happy faces,negative,Playing an action game resulted in two effects: ... and one that reflected a reduction in the allocation of attention to happy faces.,True,needs_review
252,paper_002,The effects of an action video game on visual and affective information processing,study_1,not_reported,not_reported,slow wave ERP activity (central–parietal and frontal) greater for targets than nontargets,positive,"In contrast, playing a non-action game resulted in changes in slow wave activity over the central–parietal and frontal regions that were greater for targets (i.e., angry and ha...",True,needs_review



== paper_017 ==


,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
542,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Identification of four phases in PlayStation business model evolution,not_reported,We detected that the evolution of the Sony PlayStation’s BM can be articulated in four phases where mechanisms typical of mobile games have been detected and incrementally inte...,False,needs_review
543,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Integration of mobile-game mechanisms into PlayStation's business model,not_reported,We detected that the evolution of the Sony PlayStation’s BM can be articulated in four phases where mechanisms typical of mobile games have been detected and incrementally inte...,False,needs_review
544,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Innovations coded according to value creation and value capture,not_reported,All the innovations that were identified were coded in terms of value creation and value capture mechanisms.,False,needs_review



== paper_196 ==


,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
3340,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Four Step Square Test,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",True,needs_review
3341,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Timed 25-Foot Walk,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in ... Timed 25-Foot Walk scores (DM: .15; 95% CI, −1.06 to .76; P = .75; I2 = 0%)",True,needs_review
3342,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Berg Balance Scale (BBS),positive,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum...",True,needs_review
3343,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,clinical_significance_relative_to_minimum_detectable_change,NaN,but these were not greater than the minimum detectable change reported in the literature.,True,needs_review
3344,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Four Step Square Test,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",True,needs_review
3345,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Timed 25-Foot Walk,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in ... Timed 25-Foot Walk scores (DM: .15; 95% CI, −1.06 to .76; P = .75; I2 = 0%)",True,needs_review
3346,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Berg Balance Scale (BBS),positive,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum...",True,needs_review
3347,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,clinical_significance_relative_to_minimum_detectable_change,NaN,but these were not greater than the minimum detectable change reported in the literature.,True,needs_review


## 9. Null-Game Rows

These rows are expected when a paper discusses games/gaming broadly but does not explicitly name a game that should enter the game ontology.

In [10]:
null_game_cols = ["paper_id", "paper_title", "game_id", "game_name", "mechanic_name", "outcome_canonical", "effect_direction", "review_status"]
display(df[df["game_id"].isna()][null_game_cols].head(25))

,paper_id,paper_title,game_id,game_name,mechanic_name,outcome_canonical,effect_direction,review_status
302,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,procedural pain,negative,needs_review
303,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,anxiety,negative,needs_review
304,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,caregiver anxiety,negative,needs_review
305,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,procedure length,NaN,needs_review
306,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,need for physical restraint during procedure,negative,needs_review
307,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,adverse effects,negative,needs_review
308,paper_003,Interactive video games to reduce paediatric procedural pain and anxiety: a systematic review and meta-analysis,NaN,NaN,not_reported,acceptability/satisfaction,positive,needs_review
333,paper_004,Exposure to ﬁrst-person shooter videogames is associated with multisensory temporal precision and migraine incidence,NaN,NaN,not_reported,multisensory temporal binding window (TBW) width,positive,needs_review
334,paper_004,Exposure to ﬁrst-person shooter videogames is associated with multisensory temporal precision and migraine incidence,NaN,NaN,not_reported,veridical perception rate (accuracy of perceiving actual single flash),positive,needs_review
335,paper_004,Exposure to ﬁrst-person shooter videogames is associated with multisensory temporal precision and migraine incidence,NaN,NaN,not_reported,migraine incidence/severity associated with videogame exposure,positive,needs_review


In [12]:
non_null_game_cols = ["paper_id", "paper_title", "game_id", "game_name", "mechanic_name", "outcome_canonical", "effect_direction", "review_status"]

display(
    df[df["game_id"].notna()][non_null_game_cols].tail(25)
)

,paper_id,paper_title,game_id,game_name,mechanic_name,outcome_canonical,effect_direction,review_status
5130,paper_307,Training basic laparoscopic skills using a custom-made video game,g69,Wii,not_reported,concurrent validity (correlation between game and gold standard scores),positive,needs_review
5131,paper_307,Training basic laparoscopic skills using a custom-made video game,g375,Underground,not_reported,performance score (video game),not_reported,needs_review
5132,paper_307,Training basic laparoscopic skills using a custom-made video game,g375,Underground,not_reported,concurrent validity (correlation between game and gold standard scores),positive,needs_review
5133,paper_307,Training basic laparoscopic skills using a custom-made video game,g69,Wii,not_reported,performance score (video game),not_reported,needs_review
5134,paper_307,Training basic laparoscopic skills using a custom-made video game,g69,Wii,not_reported,concurrent validity (correlation between game and gold standard scores),positive,needs_review
5135,paper_307,Training basic laparoscopic skills using a custom-made video game,g376,virtual reality simulator,Immersive Interaction,performance score (video game),not_reported,needs_review
5136,paper_307,Training basic laparoscopic skills using a custom-made video game,g376,virtual reality simulator,Multisensory Feedback,performance score (video game),not_reported,needs_review
5137,paper_307,Training basic laparoscopic skills using a custom-made video game,g376,virtual reality simulator,Movement Control,performance score (video game),not_reported,needs_review
5138,paper_307,Training basic laparoscopic skills using a custom-made video game,g376,virtual reality simulator,Immersive Interaction,concurrent validity (correlation between game and gold standard scores),positive,needs_review
5139,paper_307,Training basic laparoscopic skills using a custom-made video game,g376,virtual reality simulator,Multisensory Feedback,concurrent validity (correlation between game and gold standard scores),positive,needs_review


In [13]:
# unique named games
display(
    df[df["game_id"].notna()][["game_id", "game_name"]]
    .drop_duplicates()
    .sort_values("game_name")
    .head(50)
)

,game_id,game_name
3598,g261,"""the video game, resembling the classic arcade game 'Asteroids'"""
1453,g114,3-dimensional educational computer game (custom)
3979,g295,3DS
3089,g205,6-minute walk test
3819,g279,Active@Home exergame
1321,g105,AimLab
2441,g174,Alice: Madness Returns
3764,g274,Anarchy Online
1511,g119,Angry Bird Space
1152,g82,Angry Birds


## 10. Strongest Supported Evidence Edges

In [14]:
edge_cols = [col for col in [
    "source_node_id",
    "source_node_type",
    "target_node_id",
    "target_node_type",
    "edge_type",
    "paper_id",
    "study_id",
    "mechanic_name",
    "outcome_canonical",
    "effect_direction",
    "edge_weight",
    "source_quote",
] if col in edges.columns]

display(edges.sort_values("edge_weight", ascending=False)[edge_cols].head(25))

,source_node_id,source_node_type,target_node_id,target_node_type,edge_type,paper_id,study_id,effect_direction,edge_weight,source_quote
3214,ms_27574,MechanicSet,study_1_out_002,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_257,study_1,positive,0.225,"Both groups improved in CB&M scores, but only the VGT group increased on the UBS and TUG with a between-group significance (p < 0.05)."
2645,ms_36464,MechanicSet,paper_196_study_1_out_003,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_196,paper_196_study_1,positive,0.225,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum..."
2643,ms_36464,MechanicSet,paper_196_study_1_out_001,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_196,paper_196_study_1,NaN,0.225,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)"
2627,ms_34414,MechanicSet,study_1_out_003,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,positive,0.225,"Students in the experimental condition perceived a higher level of situational interest than their counterparts in the comparison group (p < 0.01, and η2 = 0.301)."
2626,ms_34414,MechanicSet,study_1_out_002,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,positive,0.225,"the students’ average heart rates (HRs) were in the Target-Heart-Rate-Zone (HR = 134 bpm), which was signiﬁcantly higher than the average HR (103 bpm) from the comparison condi..."
2625,ms_34414,MechanicSet,study_1_out_001,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,NaN,0.225,"students in both groups performed better on the post-test than they did on the pre-test (p < 0.001, η2 = 0.486), and their post-test scores did not differ signiﬁcantly."
2624,ms_90490,MechanicSet,study_1_out_003,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,positive,0.225,"Students in the experimental condition perceived a higher level of situational interest than their counterparts in the comparison group (p < 0.01, and η2 = 0.301)."
2623,ms_90490,MechanicSet,study_1_out_002,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,positive,0.225,"the students’ average heart rates (HRs) were in the Target-Heart-Rate-Zone (HR = 134 bpm), which was signiﬁcantly higher than the average HR (103 bpm) from the comparison condi..."
2622,ms_90490,MechanicSet,study_1_out_001,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_194,study_1,NaN,0.225,"students in both groups performed better on the post-test than they did on the pre-test (p < 0.001, η2 = 0.486), and their post-test scores did not differ signiﬁcantly."
2618,ms_70995,MechanicSet,paper_191_study_3_out_001,Outcome,MECHANIC_SET_ASSOCIATED_WITH_OUTCOME,paper_191,paper_191_study_3,positive,0.225,"An experiment conducted with pre-testing and post-testing. Participants were generally people who don’t play video games naturally, split into 2 random groups. One group play a..."


## 11. Search Helpers

In [15]:
def search_rows(term: str, columns: list[str] | None = None, n: int = 30) -> pd.DataFrame:
    columns = columns or ["paper_title", "game_name", "mechanic_name", "outcome_raw", "outcome_canonical", "source_quote"]
    mask = pd.Series(False, index=df.index)
    for col in columns:
        if col in df:
            mask = mask | df[col].astype(str).str.contains(term, case=False, na=False, regex=False)
    return df.loc[mask, spot_cols].head(n)

# Examples:
display(search_rows("Minecraft", n=10))
display(search_rows("multiple sclerosis", n=10))
display(search_rows("Sony PlayStation", n=10))

,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
476,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Cooperative Play,stereotypical views (cognitive stereotype),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
477,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Building,stereotypical views (cognitive stereotype),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
478,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Exploration,stereotypical views (cognitive stereotype),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
479,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Perspective Taking,stereotypical views (cognitive stereotype),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
480,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Cooperative Play,negative emotions (affective prejudice),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
481,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Building,negative emotions (affective prejudice),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
482,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Exploration,negative emotions (affective prejudice),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
483,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Perspective Taking,negative emotions (affective prejudice),positive,"children who participated in the intergroup program showed reduced intergroup bias on both cognitive and emotional indicators, including reduced stereotypical views, negative e...",True,needs_review
484,paper_011,Gaming for peace: Virtual contact through cooperative video gaming increases children's intergroup tolerance in the context of the Israeli–Palestinian conflict,study_1,Minecraft,Cooperative Pla

,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
3340,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Four Step Square Test,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",True,needs_review
3341,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Timed 25-Foot Walk,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in ... Timed 25-Foot Walk scores (DM: .15; 95% CI, −1.06 to .76; P = .75; I2 = 0%)",True,needs_review
3342,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Berg Balance Scale (BBS),positive,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum...",True,needs_review
3343,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,clinical_significance_relative_to_minimum_detectable_change,NaN,but these were not greater than the minimum detectable change reported in the literature.,True,needs_review
3344,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Four Step Square Test,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in Four Step Square Test (DM: −.74; 95% CI, −2.79 to 1.32; P = .48; I2 = 0%)",True,needs_review
3345,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Timed 25-Foot Walk,NaN,"We found no signiﬁcant differences between the video game therapy group and the control group in ... Timed 25-Foot Walk scores (DM: .15; 95% CI, −1.06 to .76; P = .75; I2 = 0%)",True,needs_review
3346,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,Berg Balance Scale (BBS),positive,"We did observe intergroup differences in BBS scores in favour of video game therapy (DM: 5.30; 95% CI,3.39—7.21; P < .001; I2 = 0%), but these were not greater than the minimum...",True,needs_review
3347,paper_196,Use of commercial video games to improve postural balance in patients with multiple sclerosis: A systematic review and meta-analysis of randomised controlled clinical trials,paper_196_study_1,NaN,not_reported,clinical_significance_relative_to_minimum_detectable_change,NaN,but these were not greater than the minimum detectable change reported in the literature.,True,needs_review


,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
542,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Identification of four phases in PlayStation business model evolution,not_reported,We detected that the evolution of the Sony PlayStation’s BM can be articulated in four phases where mechanisms typical of mobile games have been detected and incrementally inte...,False,needs_review
543,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Integration of mobile-game mechanisms into PlayStation's business model,not_reported,We detected that the evolution of the Sony PlayStation’s BM can be articulated in four phases where mechanisms typical of mobile games have been detected and incrementally inte...,False,needs_review
544,paper_017,Business model innovation in video-game consoles to face the threats of mobile gaming: Evidence from the case of Sony PlayStation,study_1,NaN,not_reported,Innovations coded according to value creation and value capture,not_reported,All the innovations that were identified were coded in terms of value creation and value capture mechanisms.,False,needs_review
4645,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,BMI-SDS,not_reported,"Primary outcomes are adolescents’ measured BMI-SDS (SDS = adjusted for mean standard deviation score), waist circumference-SDS, hip circumference and sum of skinfolds.",False,needs_review
4646,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,waist circumference-SDS,not_reported,"Primary outcomes are adolescents’ measured BMI-SDS (SDS = adjusted for mean standard deviation score), waist circumference-SDS, hip circumference and sum of skinfolds.",False,needs_review
4647,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,hip circumference,not_reported,"Primary outcomes are adolescents’ measured BMI-SDS (SDS = adjusted for mean standard deviation score), waist circumference-SDS, hip circumference and sum of skinfolds.",False,needs_review
4648,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,sum of skinfolds,not_reported,"Primary outcomes are adolescents’ measured BMI-SDS (SDS = adjusted for mean standard deviation score), waist circumference-SDS, hip circumference and sum of skinfolds.",False,needs_review
4649,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,time spent playing active and non-active games,not_reported,"Secondary outcomes are adolescents’ self-reported time spent playing active and non-active games, other sedentary activities and consumption of sugar-sweetened beverages.",False,needs_review
4650,paper_285,"Active video games as a tool to prevent excessive weight gain in adolescents: rationale, design and methods of a randomized controlled trial",study_1,Sony PlayStation Move,not_reported,time spent in other sedentary activities,not_reported,"Secondary outcomes are adolescents’ self-reported time spent playing active and non-active games, other sedentary activities and consumption of sugar-sweetened beverages.",False,needs_review
4651

## 12. Review Queue Samples

In [16]:
review_df = pd.DataFrame(review_items)
display(review_df.head(10))

if "reason_for_review" in review_df:
    display(review_df["reason_for_review"].value_counts().head(20).to_frame("items"))

,review_id,item_type,paper_id,study_id,current_value,suggested_value,reason_for_review,source_quote,source_page,options,status
0,rev_00001,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,Mean β SD energy expenditure was signiﬁcantly greater for all game conditions compared with rest (all P≤.01) and ranged from 1.46±.41 METs to 2.97±1.16 METs.,1.0,"[accept, reject, edit]",pending
1,rev_00002,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,"There was no signiﬁcant difference in energy expenditure, activity counts, or perceived exertion between equivalent games played while standing and seated.",1.0,"[accept, reject, edit]",pending
2,rev_00003,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,"There was no signiﬁcant difference in energy expenditure, activity counts, or perceived exertion between equivalent games played while standing and seated.",1.0,"[accept, reject, edit]",pending
3,rev_00004,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,No signiﬁcant correlations were observed between energy expenditure or activity counts and balance status.,1.0,"[accept, reject, edit]",pending
4,rev_00005,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,Mean β SD energy expenditure was signiﬁcantly greater for all game conditions compared with rest (all P≤.01) and ranged from 1.46±.41 METs to 2.97±1.16 METs.,1.0,"[accept, reject, edit]",pending
5,rev_00006,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,Mean β SD energy expenditure was signiﬁcantly greater for all game conditions compared with rest (all P≤.01) and ranged from 1.46±.41 METs to 2.97±1.16 METs.,1.0,"[accept, reject, edit]",pending
6,rev_00007,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,Mean β SD energy expenditure was signiﬁcantly greater for all game conditions compared with rest (all P≤.01) and ranged from 1.46±.41 METs to 2.97±1.16 METs.,1.0,"[accept, reject, edit]",pending
7,rev_00008,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,"There was no signiﬁcant difference in energy expenditure, activity counts, or perceived exertion between equivalent games played while standing and seated.",1.0,"[accept, reject, edit]",pending
8,rev_00009,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,"There was no signiﬁcant difference in energy expenditure, activity counts, or perceived exertion between equivalent games played while standing and seated.",1.0,"[accept, reject, edit]",pending
9,rev_00010,attribution_level,paper_001,study_1,mechanic_set_inferred,None,Row marked needs_review. Mechanic attribution is inferred.,"There was no signiﬁcant difference in energy expenditure, activity counts, or perceived exertion between equivalent games played while standing and seated.",1.0,"[accept, reject, edit]",pending


,items
reason_for_review,
Row marked needs_review. Mechanic attribution is inferred.,3964
Row marked needs_review. Unsupported or overgenerated row.,1195


## 13. Title Quality Flags

This is a lightweight title audit. It helps find rows where PDF layout leaked publisher text into the title.

In [17]:
title_df = pd.DataFrame(study_ontology.get("papers", []))
if title_df.empty:
    title_df = df[["paper_id", "paper_title"]].drop_duplicates()
else:
    if "title" in title_df.columns and "paper_title" not in title_df.columns:
        title_df = title_df.rename(columns={"title": "paper_title"})

bad_title_patterns = [
    "www.",
    "journal homepage",
    "contents lists available",
    "open access article under",
    "page 1 of",
    "vol.:(",
    "procedia -",
    "springerlink",
]

mask = pd.Series(False, index=title_df.index)
for pattern in bad_title_patterns:
    mask = mask | title_df["paper_title"].astype(str).str.contains(pattern, case=False, na=False, regex=False)

flagged_titles = title_df.loc[mask].sort_values("paper_id")
print("flagged titles:", len(flagged_titles))
display(flagged_titles.head(50))

flagged titles: 19


,paper_id,paper_title,authors,year,doi,venue,source_pdf
165,paper_166,This is an open access article under the CC BY-NC-ND license (http://creativecommons.org/licenses/by-nc-nd/4.0/). Do action video games make safer drivers? The effects of video...,[],2023,10.1016/j.trf.2023.07.006,None,1-s2.0-S1369847823001535-main.pdf
180,paper_181,Procedia - Social and Behavioral Sciences 82 ( 2013 ) 933 – 941,[],2013,10.1016/j.sbspro.2013.06.374,None,1-s2.0-S1877042813014419-main.pdf
181,paper_182,Procedia - Social and Behavioral Sciences 98 ( 2014 ) 286 – 291,[],2014,10.1016/j.sbspro.2014.03.418,None,1-s2.0-S1877042814025099-main.pdf
182,paper_183,Procedia - Social and Behavioral Sciences 98 ( 2014 ) 1738 – 1743,[],2014,10.1016/j.sbspro.2014.03.601,None,1-s2.0-S1877042814026925-main.pdf
183,paper_184,Procedia - Social and Behavioral Sciences 132 ( 2014 ) 58 – 64,[],2014,10.1016/j.sbspro.2014.04.278,None,1-s2.0-S1877042814031887-main.pdf
184,paper_185,Procedia - Social and Behavioral Sciences 149 ( 2014 ) 837 – 841,[],2014,10.1016/j.sbspro.2014.08.323,None,1-s2.0-S1877042814050368-main.pdf
185,paper_186,Procedia - Social and Behavioral Sciences 176 ( 2015 ) 419 – 424,[],2015,10.1016/j.sbspro.2015.01.491,None,1-s2.0-S1877042815005285-main.pdf
188,paper_189,This is an open access article under the CC BY-NC-ND license (http://creativecommons.org/licenses/by-nc-nd/4.0/) Peer-review under responsibility of the scientific committee of...,[],2019,10.1016/j.procs.2019.12.046,None,1-s2.0-S1877050919320575-main.pdf
189,paper_190,This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0) Peer-review under responsibility of the scientific committee of...,[],2020,10.1016/j.procs.2020.09.199,None,1-s2.0-S1877050920321013-main.pdf
190,paper_191,This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0) Peer-review under responsibility of the scientific committee of...,[],2021,10.1016/j.procs.2021.12.027,None,1-s2.0-S1877050920324698-main.pdf


## 14. Optional: Build a NetworkX Graph Object

In [18]:
import networkx as nx

G = nx.DiGraph()
for node in graph.get("nodes", []):
    node_id = node.get("node_id") or node.get("id")
    attrs = {k: v for k, v in node.items() if k not in {"node_id", "id"}}
    if node_id:
        G.add_node(node_id, **attrs)

for edge in graph.get("edges", []):
    source = edge.get("source_node_id") or edge.get("source") or edge.get("source_id")
    target = edge.get("target_node_id") or edge.get("target") or edge.get("target_id")
    attrs = {k: v for k, v in edge.items() if k not in {"source_node_id", "source", "source_id", "target_node_id", "target", "target_id"}}
    if source and target:
        G.add_edge(source, target, **attrs)

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
print("weakly connected components:", nx.number_weakly_connected_components(G) if G.number_of_nodes() else 0)

nodes: 3006
edges: 2134
weakly connected components: 2135


## 15. Optional: Export a Small Manual-Audit CSV

In [19]:
audit_sample = pd.concat(
    [
        df[df["is_supported_evidence"].astype(bool)].sample(min(25, int(df["is_supported_evidence"].astype(bool).sum())), random_state=7),
        df[df["review_status"].eq("needs_review")].sample(min(25, int(df["review_status"].eq("needs_review").sum())), random_state=7),
    ],
    ignore_index=True,
).drop_duplicates()

audit_path = DATA_DIR / "manual_audit_sample.csv"
audit_sample.to_csv(audit_path, index=False)
print("Wrote", audit_path)
display(audit_sample[spot_cols].head(20))

Wrote /Users/srishtisaha/Desktop/infinite_lives/game-evidence-graph/data/processed/manual_audit_sample.csv


,paper_id,paper_title,study_id,game_name,mechanic_name,outcome_canonical,effect_direction,source_quote,is_supported_evidence,review_status
0,paper_094,Sex role stereotyping is hard to kill: A ﬁeld experiment measuring social responses to user characteristics and behavior in an online multiplayer ﬁrst-person shooter game,paper_094_study_1,online first-person shooter video game,Rapid Response,compliance with friend requests,positive,"A 2 × 3 × 2 virtual ﬁeld experiment (N = 520) in an online ﬁrst-person shooter video game examined effects of a confederate players’ sex, communication style, and skill on play...",True,needs_review
1,paper_147,Theory of consumption value: A lens to examine the use and continual use intention of online game subscription services,study_1,Microsoft’s Xbox Game Pass,not_reported,Perceived playing-games value,positive,"Three value types (playing games, enjoyment, and price) contribute to the use (continual use) intention.",True,needs_review
2,paper_171,Physiological response during activity programs using Wii-based video games in patients with cystic ﬁbrosis (CF),study_1,mat controller,not_reported,Wii-Acti and Wii-Train impose metabolic demand comparable to 6MWT,positive,Wii-Acti and Wii-Train impose a signiﬁcantly high metabolic demand comparable to the 6MWT.,True,needs_review
3,paper_171,Physiological response during activity programs using Wii-based video games in patients with cystic ﬁbrosis (CF),study_1,EA SPORTS™ACTIVE,Movement Control,compliance / adherence,positive,All patients were compliant with all three Wii™modalities.,True,needs_review
4,paper_288,Active video games: the mediating effect of aerobic fitness on body composition,study_1,Dance Factory,not_reported,Body mass index (BMI) change from baseline at 24 weeks,positive,"For change in BMI, the observed difference between the two treatment groups reduced from −0.3265 (p = 0.01) to −0.2992 (p = 0.02), or approximately 8% when aerobic fitness was ...",True,needs_review
5,paper_133,The Fortnite social paradox: The effects of violent-cooperative multi-player video games on children’s basic psychological needs and prosocial behavior,paper_133_study1,a violent multiplayer online game,not_reported,"satisfaction of psychological needs (competence, relatedness, positive-emotions, enjoyment) as mediator",positive,"Satisfaction of the psychological needs of competence, relatedness, positive-emotions, and enjoyment was examined as possible mediator of the hypothesized effect of cooperative...",True,needs_review
6,paper_065,The effect of simulation games on the learning of computational problem solving,study_1,NaN,not_reported,analytical reasoning strategy use,positive,"The students who perceived a ﬂow experience state frequently applied trial-and-error, learning-by-example, and analytical reasoning strategies to learn the computational proble...",True,needs_review
7,paper_025,Moderating effects of visual attention and action video game play on perceptual learning with the texture discrimination task,paper_025_exp1,action video games,Targeting,TDT baseline performance (task threshold/accuracy),positive,"We examined baseline performance on the TDT, learning on the initially trained TDT stimuli and transfer to a subsequently trained background orientation.",True,needs_review
8,paper_263,Effects of interactive video-game–based exercise on balance in older adults with mild-to-moderate Parkinson’s disease,paper_263_study_1,step mat,not_reported,multi-directional reach distance / dynamic reach (Multi-Directional Reach Test subscales),positive,"After Bonferroni adjustment, the changes in ... two subscales of Multi-Directional Reach Test were significantly different between two groups in the first 6-week period.",True,needs_review
9,paper_166,This is an open access article under the CC BY-NC-ND license (http://creativecommons.org/licenses/by-nc-nd/4.0/). Do action video games make safer drivers? The effects of video...,study_1,action video games,Combat,spare